# cv-data-lesson OCR実モデル受け入れテスト

授業用Notebookとは別の確認用Notebookです。新しいCPUランタイムで固定コミットのライブラリとCPU版PaddlePaddleを導入し、固定revisionのPP-OCRv5 mobileを実際に読み込みます。PNG・JPEG・WebP・Pillow Image入力、壊れた画像のエラー、公開済み実レシート画像を確認します。公開実レシートは非商用利用に限り、画像や読み取り結果を共有しないでください。

In [ ]:
import sys
print("Python:", sys.version)
assert (3, 11) <= sys.version_info[:2] < (3, 14), "Python 3.11〜3.13のランタイムを選択してください"

## CPU版PaddlePaddleと固定コミットを導入

Hugging Faceの公開モデルはログイン不要です。`HF_TOKEN`の警告が出ても、今回の動作確認は継続できます。依存関係の競合を避けるため、OCR専用の新しいランタイムで実行してください。

In [ ]:
%pip install -q "paddlepaddle==3.2.0" -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
%pip install -q "cv-data-lesson[ocr,dataframe] @ git+https://github.com/708san/CV_data.git@5b3a2425eaf36caac0a0fd87893bd78d75a250da"

In [ ]:
from pathlib import Path
from importlib.metadata import version
from PIL import Image, ImageDraw, ImageFont
from cvlesson import datasets, ocr

print("cv-data-lesson:", version("cv-data-lesson"))
print("PaddlePaddle:", version("paddlepaddle"))
print("OCR devices:", ocr.available_devices())
assert ocr.available_devices()["cpu"] is True
OCR_CACHE = Path("/content/cvlesson-ocr-model-cache")
DATA_CACHE = Path("/content/cvlesson-ocr-data-cache")

## OCRモデルを読み込む

このセルで固定revisionの検出モデルと認識モデルを取得します。初回だけ時間がかかります。

In [ ]:
reader = ocr.load(device="cpu", cache_dir=OCR_CACHE)
assert reader.device == "cpu"
print("固定OCRモデルの読み込み: OK")

## PNG・JPEG・WebP・Pillow Imageを実際に読む

合成した文字画像を3形式で保存します。WebPを渡しても内部で一時PNGへデコードされ、空結果を返さず、通常のOCR結果として扱えることを確認します。

In [ ]:
synthetic_image = Image.new("RGB", (1200, 260), "white")
draw = ImageDraw.Draw(synthetic_image)
font_path = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
font = ImageFont.truetype(font_path, 92) if Path(font_path).exists() else ImageFont.load_default()
draw.text((40, 65), "TOTAL 1280 JPY", fill="black", font=font)
synthetic_dir = Path("/content/cvlesson-ocr-inputs")
synthetic_dir.mkdir(parents=True, exist_ok=True)
png_path = synthetic_dir / "sample.png"
jpeg_path = synthetic_dir / "sample.jpg"
webp_path = synthetic_dir / "sample.webp"
synthetic_image.save(png_path, format="PNG")
synthetic_image.save(jpeg_path, format="JPEG")
synthetic_image.save(webp_path, format="WEBP")
png_result = reader.predict(png_path)
jpeg_result = reader.predict(jpeg_path)
webp_result = reader.predict(webp_path)
pillow_result = reader.predict(synthetic_image)
for result in (png_result, jpeg_result, webp_result, pillow_result):
    assert isinstance(result.text, str)
    for line in result.lines:
        assert line.text.strip()
        assert 0.0 <= line.confidence <= 1.0
assert webp_result.text.strip(), "WebPが空結果になっています"
print("PNG/JPEG/WebP/Pillow Image OCR: OK")
print("WebPの読み取り行数:", len(webp_result.lines))

In [ ]:
broken_path = synthetic_dir / "broken.webp"
broken_path.write_bytes(b"not an image")
try:
    reader.predict(broken_path)
except ocr.OCRError as error:
    print("壊れた画像のエラー: OK -", error)
else:
    raise AssertionError("壊れた画像が空のOCR結果として処理されました")

## 固定Releaseの日本語レシート画像で確認

CC BY-NC 4.0の固定artifactから1枚だけ取得します。注釈は画像内の一部であり、OCRの全文正解ではありません。店舗名・取引番号・QRコードなどが含まれる可能性があるため、結果を公開しないでください。

In [ ]:
receipt_data = datasets.get("japanese_receipts_detailed", cache_dir=DATA_CACHE)
receipt_frame = receipt_data.to_df()
receipt_sample = receipt_frame.iloc[0]
receipt_path = Path(receipt_sample["image_path"])
assert receipt_path.is_file()
receipt_result = reader.predict(receipt_path)
assert isinstance(receipt_result.text, str)
print("日本語レシート実画像OCR: OK")
print("画像数:", len(receipt_frame), " / 読み取り行数:", len(receipt_result.lines))

## 判定

ここまで成功すれば、固定revisionのPaddleOCRモデルをCPUで読み込み、対応画像形式を実際に推論し、WebPを空結果にせず、壊れた画像を`OCRError`として扱うことを確認できています。読み取り精度は、元画像・部分注釈と人が照合してください。